# AMP Challenge — Step 4: External Known-AMP Near-Match Scoring

Pipeline stage: **Step 3 clean pool → Step 4 external near-match features → later physchem/activity/safety/diversity scoring**.

This notebook scores near matches to external known-AMP databases without introducing a new hard rejection threshold.


In [1]:
# ============================================================
# AMP Challenge — Step 4
# External Known-AMP Near-Match Scoring
# ============================================================
#
# INPUT:
#   The ALL_CLEAN.csv produced by Step 3
#   (e.g. MULTIBATCH_5__...__ALL_CLEAN.csv)
#
# EXTERNAL REFERENCES:
#   DBAASP.fasta
#   dbAMP3.fasta
#   ADP6.fasta / APD6.fasta
#   DRAMP.fasta (optional)
#
# PURPOSE:
#   Add near-match information for known AMP databases WITHOUT
#   deleting candidates solely because they are near matches.
#
# IMPORTANT:
#   - This is a diagnostic / ranking feature stage.
#   - It is NOT the official seqme novelty score.
#   - It does NOT apply a new arbitrary hard rejection threshold.
#   - Exact overlaps are flagged if present.
#   - Similarity here uses RapidFuzz fuzz.ratio after a fast
#     k-mer candidate-retrieval step, so it is an approximate
#     nearest-known-AMP search suitable for a large ~250K pool.
#
# OUTPUT:
#   ALL_CLEAN_WITH_EXTERNAL_NEARMATCH.csv
#   EXTERNAL_NEARMATCH_SUMMARY.csv
#   STEP4_MANIFEST.txt
# ============================================================

!pip -q install rapidfuzz pandas tqdm

from google.colab import files
from rapidfuzz import fuzz
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from datetime import datetime
import pandas as pd
import numpy as np
import os
import re
import heapq

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

KMER_K = 3
TOP_CANDIDATES_PER_DB = 40
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# These are descriptive flags only — NOT competition rejection rules.
NEAR_EXACT_FLAG = 90.0
HIGH_SIMILARITY_FLAG = 80.0

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def clean_name(filename):
    name = os.path.basename(filename)
    for ext in [".fasta", ".fa", ".faa", ".fas", ".txt"]:
        if name.lower().endswith(ext):
            name = name[:-len(ext)]
            break
    return re.sub(r"[^A-Za-z0-9_-]+", "_", name).strip("_")

def normalize_db_name(filename):
    x = clean_name(filename).lower()
    if "dbaasp" in x:
        return "DBAASP"
    if "dbamp" in x:
        return "dbAMP3"
    if "apd" in x or "adp" in x:
        return "APD6"
    if "dramp" in x:
        return "DRAMP"
    return clean_name(filename)

def read_fasta(filename):
    records = []
    current_id = None
    seq_parts = []

    def flush():
        nonlocal current_id, seq_parts
        if current_id is not None:
            seq = "".join(seq_parts).upper().strip()
            if seq:
                records.append((current_id, seq))

    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if line.startswith(">"):
                flush()
                current_id = line[1:].strip() or f"unnamed_{len(records)+1}"
                seq_parts = []
            elif line:
                seq_parts.append(re.sub(r"\s+", "", line))
        flush()

    return records

def kmers(seq, k=3):
    if len(seq) < k:
        return {seq}
    return {seq[i:i+k] for i in range(len(seq)-k+1)}

def build_db_index(records, k=3):
    """
    Build:
      seqs: unique reference sequences
      ids: first ID associated with each unique sequence
      inv: kmer -> list of reference indices
      exact: sequence -> reference index
    """
    seqs = []
    ids = []
    exact = {}
    inv = defaultdict(list)

    for rid, seq in records:
        seq = seq.upper().strip()
        if not seq or seq in exact:
            continue
        idx = len(seqs)
        exact[seq] = idx
        seqs.append(seq)
        ids.append(rid)

    for idx, seq in enumerate(seqs):
        for token in kmers(seq, k):
            inv[token].append(idx)

    return {
        "seqs": seqs,
        "ids": ids,
        "exact": exact,
        "inv": inv,
    }

def shortlist_candidates(query, db, k=3, top_n=40):
    """
    Fast candidate retrieval using shared unique k-mers.
    Returns reference indices likely to be similar.
    """
    counts = Counter()
    qkmers = kmers(query, k)

    for token in qkmers:
        for idx in db["inv"].get(token, []):
            counts[idx] += 1

    if not counts:
        return []

    # Prefer references sharing the most query k-mers.
    return [idx for idx, _ in heapq.nlargest(top_n, counts.items(), key=lambda x: x[1])]

def best_near_match(query, db, k=3, top_n=40):
    """
    Returns best approximate near match within k-mer-shortlisted candidates.
    Exact matches are guaranteed to be detected separately.
    """
    if query in db["exact"]:
        idx = db["exact"][query]
        return 100.0, db["ids"][idx], db["seqs"][idx], True

    candidates = shortlist_candidates(query, db, k=k, top_n=top_n)

    if not candidates:
        return 0.0, None, None, False

    best_score = -1.0
    best_idx = None

    for idx in candidates:
        score = float(fuzz.ratio(query, db["seqs"][idx]))
        if score > best_score:
            best_score = score
            best_idx = idx

    return (
        round(best_score, 3),
        db["ids"][best_idx],
        db["seqs"][best_idx],
        False
    )

# ------------------------------------------------------------
# 1. UPLOAD STEP-3 ALL_CLEAN CSV
# ------------------------------------------------------------

print("\n==========================================")
print("UPLOAD STEP-3 ALL_CLEAN CSV")
print("==========================================")
print("Upload the clean CSV produced by MultiBatch Clean Pool Builder.\n")

clean_upload = files.upload()
clean_files = list(clean_upload.keys())

if len(clean_files) != 1:
    raise ValueError("Please upload exactly ONE ALL_CLEAN.csv file.")

clean_csv = clean_files[0]
df = pd.read_csv(clean_csv)

if "sequence" not in df.columns:
    raise ValueError("Input CSV must contain a column named 'sequence'.")

df["sequence"] = df["sequence"].astype(str).str.upper().str.strip()

print(f"\nLoaded clean candidates: {len(df):,}")
print("Input:", clean_csv)

# ------------------------------------------------------------
# 2. UPLOAD EXTERNAL KNOWN-AMP FASTA FILES
# ------------------------------------------------------------

print("\n==========================================")
print("UPLOAD EXTERNAL KNOWN-AMP DATABASES")
print("==========================================")
print(
    "Upload DBAASP, dbAMP3, APD6/ADP6 and optionally DRAMP.\n"
    "Do NOT upload antibacterial.fasta here; the official >80% hard "
    "screen was already handled in Step 3.\n"
)

ref_upload = files.upload()
ref_files = list(ref_upload.keys())

if not ref_files:
    raise ValueError("No external reference FASTA was uploaded.")

# Reject accidental official reference upload to keep Step 4 conceptually clean.
bad_official = [f for f in ref_files if "antibacterial" in f.lower()]
if bad_official:
    raise ValueError(
        "Remove antibacterial.fasta from Step 4. "
        "This step is only for external known-AMP near-match scoring."
    )

# ------------------------------------------------------------
# 3. BUILD EXTERNAL DATABASE INDICES
# ------------------------------------------------------------

dbs = {}

print("\n==========================================")
print("REFERENCE DATABASE SUMMARY")
print("==========================================")

for filename in ref_files:
    db_name = normalize_db_name(filename)
    records = read_fasta(filename)
    db = build_db_index(records, k=KMER_K)
    dbs[db_name] = db

    print(
        f"{db_name:12s} | "
        f"records={len(records):7d} | "
        f"unique={len(db['seqs']):7d}"
    )

print("\nDatabases used:", ", ".join(dbs.keys()))

# ------------------------------------------------------------
# 4. SCORE EVERY CLEAN CANDIDATE
# ------------------------------------------------------------

result_rows = []

for row in tqdm(
    df.itertuples(index=False),
    total=len(df),
    desc="External near-match scoring"
):
    seq = str(row.sequence).upper().strip()

    per_db = {}
    global_best = {
        "score": -1.0,
        "database": None,
        "match_id": None,
        "match_sequence": None,
        "exact": False,
    }

    for db_name, db in dbs.items():
        score, match_id, match_seq, exact = best_near_match(
            seq,
            db,
            k=KMER_K,
            top_n=TOP_CANDIDATES_PER_DB
        )

        per_db[db_name] = {
            "score": score,
            "match_id": match_id,
            "match_sequence": match_seq,
            "exact": exact
        }

        if score > global_best["score"]:
            global_best = {
                "score": score,
                "database": db_name,
                "match_id": match_id,
                "match_sequence": match_seq,
                "exact": exact,
            }

    # Add original columns first.
    out = row._asdict()

    # Add per-database columns.
    for db_name in dbs.keys():
        safe = re.sub(r"[^A-Za-z0-9]+", "_", db_name).strip("_")
        out[f"{safe}_best_similarity_pct"] = per_db[db_name]["score"]
        out[f"{safe}_best_match_id"] = per_db[db_name]["match_id"]
        out[f"{safe}_best_match_sequence"] = per_db[db_name]["match_sequence"]
        out[f"{safe}_exact_overlap"] = per_db[db_name]["exact"]

    # Global external near-match summary.
    out["external_max_similarity_pct"] = round(global_best["score"], 3)
    out["closest_external_database"] = global_best["database"]
    out["closest_external_match_id"] = global_best["match_id"]
    out["closest_external_match_sequence"] = global_best["match_sequence"]
    out["external_exact_overlap"] = bool(global_best["exact"])

    # Flags are descriptive only; they do NOT remove a peptide.
    out["near_exact_flag_ge90"] = global_best["score"] >= NEAR_EXACT_FLAG
    out["high_similarity_flag_ge80"] = global_best["score"] >= HIGH_SIMILARITY_FLAG

    # Simple monotonic diagnostic feature:
    # 100 = very dissimilar to the closest known AMP under this approximate search
    #   0 = exact match
    # NOT an official seqme score.
    out["external_novelty_diagnostic"] = round(
        max(0.0, 100.0 - global_best["score"]), 3
    )

    result_rows.append(out)

scored_df = pd.DataFrame(result_rows)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

summary_rows = []

for db_name in dbs.keys():
    safe = re.sub(r"[^A-Za-z0-9]+", "_", db_name).strip("_")
    sim_col = f"{safe}_best_similarity_pct"
    exact_col = f"{safe}_exact_overlap"

    summary_rows.append({
        "database": db_name,
        "candidates": len(scored_df),
        "exact_overlaps": int(scored_df[exact_col].sum()),
        "similarity_ge_90": int((scored_df[sim_col] >= 90).sum()),
        "similarity_ge_80": int((scored_df[sim_col] >= 80).sum()),
        "median_best_similarity": round(float(scored_df[sim_col].median()), 3),
        "mean_best_similarity": round(float(scored_df[sim_col].mean()), 3),
        "max_best_similarity": round(float(scored_df[sim_col].max()), 3),
    })

summary_df = pd.DataFrame(summary_rows)

print("\n")
print("=" * 72)
print("STEP 4 — EXTERNAL KNOWN-AMP NEAR-MATCH SUMMARY")
print("=" * 72)
print(f"Candidates scored                 : {len(scored_df):,}")
print(f"External databases                : {len(dbs):,}")
print(
    f"Candidates with external >=90%    : "
    f"{int(scored_df['near_exact_flag_ge90'].sum()):,}"
)
print(
    f"Candidates with external >=80%    : "
    f"{int(scored_df['high_similarity_flag_ge80'].sum()):,}"
)
print(
    f"Exact external overlaps            : "
    f"{int(scored_df['external_exact_overlap'].sum()):,}"
)

display(summary_df)

print("\nGlobal external similarity distribution:")
display(
    scored_df["external_max_similarity_pct"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("value")
)

# ------------------------------------------------------------
# 6. SAVE OUTPUTS
# ------------------------------------------------------------

output_csv = "ALL_CLEAN_WITH_EXTERNAL_NEARMATCH.csv"
summary_csv = "EXTERNAL_NEARMATCH_SUMMARY.csv"
manifest_txt = "STEP4_EXTERNAL_NEARMATCH_MANIFEST.txt"

scored_df.to_csv(output_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

with open(manifest_txt, "w") as f:
    f.write("AMP Challenge — Step 4: External Known-AMP Near-Match Scoring\n")
    f.write("=" * 72 + "\n\n")
    f.write(f"Timestamp: {datetime.now().isoformat(timespec='seconds')}\n")
    f.write(f"Input clean CSV: {clean_csv}\n")
    f.write(f"Candidates scored: {len(scored_df)}\n\n")

    f.write("External databases:\n")
    for filename in ref_files:
        f.write(f"  - {filename}\n")

    f.write("\nParameters:\n")
    f.write(f"  KMER_K={KMER_K}\n")
    f.write(f"  TOP_CANDIDATES_PER_DB={TOP_CANDIDATES_PER_DB}\n")
    f.write(f"  NEAR_EXACT_FLAG={NEAR_EXACT_FLAG}\n")
    f.write(f"  HIGH_SIMILARITY_FLAG={HIGH_SIMILARITY_FLAG}\n\n")

    f.write("Method note:\n")
    f.write(
        "This stage uses unique 3-mer overlap to retrieve a small shortlist "
        "of likely neighboring reference peptides, then refines those hits "
        "with RapidFuzz fuzz.ratio. Therefore the nearest-match search is "
        "approximate and is intended as a scalable diagnostic/ranking feature, "
        "not as the official seqme novelty implementation.\n"
    )
    f.write(
        "No peptide is removed solely because of near-match similarity in "
        "this stage. The resulting similarity features will later be combined "
        "with activity, safety, physicochemical and diversity information.\n"
    )

print("\nOUTPUT FILES")
print(" -", output_csv)
print(" -", summary_csv)
print(" -", manifest_txt)

# ------------------------------------------------------------
# 7. DOWNLOAD
# ------------------------------------------------------------

for x in [output_csv, summary_csv, manifest_txt]:
    files.download(x)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.9 MB/s eta 0:00:00

UPLOAD STEP-3 ALL_CLEAN CSV
Upload the clean CSV produced by MultiBatch Clean Pool Builder.



Saving MULTIBATCH_5__filtered_against__ADP6__DBAASP__antibacterial__dbAMP3__ALL_CLEAN.csv to MULTIBATCH_5__filtered_against__ADP6__DBAASP__antibacterial__dbAMP3__ALL_CLEAN.csv

Loaded clean candidates: 246,795
Input: MULTIBATCH_5__filtered_against__ADP6__DBAASP__antibacterial__dbAMP3__ALL_CLEAN.csv

UPLOAD EXTERNAL KNOWN-AMP DATABASES
Upload DBAASP, dbAMP3, APD6/ADP6 and optionally DRAMP.
Do NOT upload antibacterial.fasta here; the official >80% hard screen was already handled in Step 3.



Saving ADP6.fasta to ADP6.fasta
Saving DBAASP.fasta to DBAASP.fasta
Saving dbAMP3.fasta to dbAMP3.fasta

REFERENCE DATABASE SUMMARY
APD6         | records=   2580 | unique=   2580
DBAASP       | records=   1976 | unique=   1762
dbAMP3       | records=  35599 | unique=  35471

Databases used: APD6, DBAASP, dbAMP3


External near-match scoring:   0%|          | 0/246795 [00:00<?, ?it/s]



STEP 4 — EXTERNAL KNOWN-AMP NEAR-MATCH SUMMARY
Candidates scored                 : 246,795
External databases                : 3
Candidates with external >=90%    : 0
Candidates with external >=80%    : 461
Exact external overlaps            : 0


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>